# 09 — Tensors and Numerical Python

**Master syllabus coverage:** V2 2.1

## Why this matters

Nearly every SLM bug eventually appears as a wrong shape, dtype, device, or interpretation of an axis. Tensor literacy is the runtime type system of ML.

## Learning objectives

- Read and write scalar, vector, matrix, sequence, and batch shapes.
- Choose appropriate integer, floating-point, and boolean dtypes.
- Use indexing, reshaping, reductions, and device movement deliberately.
- Recognize memory sharing between NumPy and CPU PyTorch tensors.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Tensor vocabulary

A tensor is an n-dimensional rectangular array plus metadata. The core metadata is:

- **shape:** length of every axis;
- **rank / ndim:** number of axes (not matrix rank);
- **dtype:** numeric representation and precision;
- **device:** where storage and operations live;
- **stride:** how indices map to memory offsets.

SLM convention used throughout the course:

- $B$: batch size, $T$: sequence length, $C$: model width;
- $V$: vocabulary size, $H$: attention heads, $D=C/H$: head width.


In [ ]:
import numpy as np
import torch

np.random.seed(42)
torch.manual_seed(42)

scalar = torch.tensor(7.0)                         # []
vector = torch.tensor([1.0, 2.0, 3.0])             # [C]
matrix = torch.arange(12).reshape(3, 4)             # [T, C]
sequence_batch = torch.randn(2, 3, 4)              # [B, T, C]

for name, value in {"scalar": scalar, "vector": vector, "matrix": matrix,
                    "sequence_batch": sequence_batch}.items():
    print(f"{name:15} shape={tuple(value.shape)!s:10} ndim={value.ndim} dtype={value.dtype}")

assert scalar.shape == torch.Size([])
assert sequence_batch.shape == (2, 3, 4)


## 2. Axes carry meaning

Shape equality is not semantic equality. `[B, T, C]` and `[T, B, C]` contain the
same number of values but mean different things. Name shapes in comments and assert
boundaries. In TypeScript terms, a tensor shape behaves like crucial runtime schema
information that the basic tensor type does not encode.


In [ ]:
B, T, C = sequence_batch.shape
per_token_mean = sequence_batch.mean(dim=-1)       # [B, T]
per_sequence_mean = sequence_batch.mean(dim=1)     # [B, C]
transposed = sequence_batch.transpose(0, 1)        # [T, B, C]

print("per token:", per_token_mean.shape)
print("per sequence:", per_sequence_mean.shape)
print("transposed:", transposed.shape)
assert per_token_mean.shape == (B, T)
assert per_sequence_mean.shape == (B, C)


## 3. Dtypes and numerical meaning

Integer tensors usually hold token IDs or labels. Floating tensors hold parameters,
activations, gradients, and probabilities. Boolean tensors hold masks. The dtype
changes range, precision, memory, and which operations are legal.

A float with $n$ elements uses approximately `n × element_size` bytes. Casting can
be lossy; integer division, overflow, and low-precision accumulation deserve explicit
attention.


In [ ]:
token_ids = torch.tensor([[1, 4, 2], [3, 0, 0]], dtype=torch.long)  # [B, T]
padding_mask = token_ids != 0                                      # [B, T]
activations = torch.randn(2, 3, 8, dtype=torch.float32)             # [B, T, C]

for tensor in (token_ids, padding_mask, activations, activations.half()):
    mib = tensor.numel() * tensor.element_size() / 2**20
    print(tensor.dtype, tensor.shape, f"{mib:.6f} MiB")

rounded = torch.tensor([16_777_216.0], dtype=torch.float32)
print("float32 loses this +1:", rounded + 1 == rounded)
assert token_ids.dtype == torch.int64 and padding_mask.dtype == torch.bool


## 4. NumPy ↔ PyTorch and device placement

NumPy is ideal for visible numerical algorithms. PyTorch adds devices, automatic
differentiation, neural-network layers, and optimized kernels. On CPU, conversion can
share memory; use `.copy()` or `.clone()` when aliasing would be surprising.


In [ ]:
array = np.arange(6, dtype=np.float32).reshape(2, 3)
tensor = torch.from_numpy(array)  # shares CPU memory
array[0, 0] = 99.0
assert tensor[0, 0].item() == 99.0

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

moved = tensor.to(device)
print("selected device:", device, "tensor device:", moved.device)
print("NumPy shares CPU storage:", tensor.data_ptr() == torch.from_numpy(array).data_ptr())


## 5. Indexing, reshaping, and reductions

Indexing removes or selects axes. Reshaping reorganizes axis boundaries while keeping
the number of elements constant. Reductions remove axes unless `keepdim=True`. Keeping
a reduced dimension often makes later broadcasting easier to reason about.


In [ ]:
x = torch.arange(2 * 3 * 4).reshape(2, 3, 4)  # [B=2, T=3, C=4]
first_sequence = x[0]                         # [T, C]
last_feature = x[:, :, -1]                    # [B, T]
flattened_tokens = x.reshape(-1, 4)           # [B*T, C]
centered = x.float() - x.float().mean(dim=-1, keepdim=True)

assert first_sequence.shape == (3, 4)
assert flattened_tokens.shape == (6, 4)
torch.testing.assert_close(centered.mean(dim=-1), torch.zeros(2, 3), atol=1e-6, rtol=0)
print("all shape and centering invariants passed")


## Failure modes to recognize

- **Axis confusion:** a valid operation produces semantically wrong results.
- **Dtype mismatch:** token IDs become floats or a floating operation truncates to integers.
- **Device mismatch:** operands on CPU and MPS/CUDA cannot participate in one operation.
- **Unexpected aliasing:** modifying a NumPy array silently changes a shared tensor.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Create logits with shape `[B=2, T=5, V=11]` and extract the final time step as `[B, V]`.
2. Compute the memory in MiB for a hypothetical `[8, 1024, 768]` float16 activation.
3. Intentionally swap batch and time, then write an assertion that catches the mistake.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can annotate every tensor in a small sequence computation and predict the result of indexing or reducing each axis.

**Next:** 10 — Broadcasting and vectorization.
